In [2]:

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Delivery Capstone - In-Memory (No JSON Ingestion)") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()



In [ ]:
delivery_data = [
("DLV001","Delhi ","D001","Delivered","120","2024-01-05 10:30"),
("DLV002","Mumbai","D002","Delivered","90","05/01/2024 11:00"),
("DLV003","Bangalore","D003","In Transit","200","2024/01/06 09:45"),
("DLV004","Delhi","D004","Cancelled","","2024-01-07 14:00"),
("DLV005","Chennai","D002","Delivered","invalid","2024-01-08 16:20"),
("DLV006","Mumbai","D005","Delivered",None,"2024-01-08 18:10"),
("DLV007","Delhi","D001","Delivered","140","09-01-2024 12:30"),
("DLV008","Bangalore","D003","Delivered","160","2024-01-09 15:45"),
("DLV009","Mumbai","D004","Delivered","110","2024-01-10 13:20"),
("DLV009","Mumbai","D004","Delivered","110","2024-01-10 13:20")
]

driver_data = [
    ("D001","Ravi","Senior"),
    ("D002","Amit","Junior"),
    ("D003","Sneha","Senior"),
    ("D004","Karan","Junior"),
    ("D005","Neha","Senior"),
]

city_zone_data = [
    ("Delhi","North"),
    ("Mumbai","West"),
    ("Bangalore","South"),
    ("Chennai","South"),
]


Phase 1

In [4]:

delivery_schema = StructType([
    StructField("delivery_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("driver_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("delivery_time_minutes", StringType(), True),  # parse later
    StructField("delivery_timestamp", StringType(), True),     # parse later
])


In [3]:

driver_schema = StructType([
    StructField("driver_id", StringType(), True),
    StructField("driver_name", StringType(), True),
    StructField("driver_level", StringType(), True),
])


In [5]:

city_zone_schema = StructType([
    StructField("city", StringType(), True),
    StructField("zone", StringType(), True),
])


Load raw delivery data

In [20]:
delivery_data = [
("DLV001","Delhi ","D001","Delivered","120","2024-01-05 10:30"),
("DLV002","Mumbai","D002","Delivered","90","05/01/2024 11:00"),
("DLV003","Bangalore","D003","In Transit","200","2024/01/06 09:45"),
("DLV004","Delhi","D004","Cancelled","","2024-01-07 14:00"),
("DLV005","Chennai","D002","Delivered","invalid","2024-01-08 16:20"),
("DLV006","Mumbai","D005","Delivered",None,"2024-01-08 18:10"),
("DLV007","Delhi","D001","Delivered","140","09-01-2024 12:30"),
("DLV008","Bangalore","D003","Delivered","160","2024-01-09 15:45"),
("DLV009","Mumbai","D004","Delivered","110","2024-01-10 13:20"),
("DLV009","Mumbai","D004","Delivered","110","2024-01-10 13:20")
]

driver_data = [
    ("D001","Ravi","Senior"),
    ("D002","Amit","Junior"),
    ("D003","Sneha","Senior"),
    ("D004","Karan","Junior"),
    ("D005","Neha","Senior"),
]

city_zone_data = [
    ("Delhi","North"),
    ("Mumbai","West"),
    ("Bangalore","South"),
    ("Chennai","South"),
]

delivery_raw = spark.createDataFrame(delivery_data, schema=delivery_schema)
driver_raw = spark.createDataFrame(driver_data, schema=driver_schema)
city_zone_raw = spark.createDataFrame(city_zone_data, schema=city_zone_schema)

Identify and flag corrupt records

In [8]:

delivery_marked = delivery_raw.withColumn(
    "time_str_norm", F.trim(F.col("delivery_time_minutes"))
).withColumn(
    "time_is_invalid",
    (F.col("time_str_norm").isNull()) |
    (F.col("time_str_norm") == "") |
    (F.lower(F.col("time_str_norm")) == "invalid") |
    (~F.col("time_str_norm").rlike("^[0-9]+$"))
)
corrupt_time_records = delivery_marked.filter(F.col("time_is_invalid"))


Validate schema correctness

In [9]:

delivery_raw.printSchema()
driver_raw.printSchema()
city_zone_raw.printSchema()


root
 |-- delivery_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- delivery_time_minutes: string (nullable = true)
 |-- delivery_timestamp: string (nullable = true)

root
 |-- driver_id: string (nullable = true)
 |-- driver_name: string (nullable = true)
 |-- driver_level: string (nullable = true)

root
 |-- city: string (nullable = true)
 |-- zone: string (nullable = true)



PHASE 2 — DATA CLEANING & STANDARDIZATION

In [10]:

clean1 = delivery_marked.select(
    F.trim(F.col("delivery_id")).alias("delivery_id"),
    F.upper(F.trim(F.col("city"))).alias("city"),            # UPPER for joins/partitioning
    F.trim(F.col("driver_id")).alias("driver_id"),
    F.trim(F.col("status")).alias("status_raw"),
    F.col("time_str_norm").alias("time_str"),
    F.trim(F.col("delivery_timestamp")).alias("timestamp_str")
)


In [11]:

# 6. Standardize status values (prefix-based mapping: DELIVERED / IN TRANSIT / CANCELLED)
status_norm = F.upper(F.col("status_raw"))
clean2 = clean1.withColumn(
    "status",
    F.when(status_norm.startswith("DEL"), F.lit("DELIVERED"))
     .when(status_norm.startswith("IN"), F.lit("IN TRANSIT"))
     .when(status_norm.startswith("CAN"), F.lit("CANCELLED"))
     .otherwise(F.lit("UNKNOWN"))
).drop("status_raw")


In [12]:

# 7. Convert delivery_time_minutes to IntegerType
clean3 = clean2.withColumn(
    "delivery_time_minutes",
    F.when(F.col("time_str").rlike("^[0-9]+$"), F.col("time_str").cast(IntegerType()))
     .otherwise(F.lit(None).cast(IntegerType()))
).drop("time_str")


In [13]:

# 8. Handle invalid and null delivery times
time_null_df = clean3.filter(F.col("delivery_time_minutes").isNull())
clean4 = clean3.filter(F.col("delivery_time_minutes").isNotNull())


In [40]:

# 9. Parse multiple timestamp formats into TimestampType with regex validation
parsed_ts = F.coalesce(
    F.when(F.col("timestamp_str").rlike("^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}$|^\\d{4}/\\d{2}/\\d{2} \\d{2}:\\d{2}$|^\\d{2}/\\d{2}/\\d{4} \\d{2}:\\d{2}$"),
        F.try_to_timestamp(F.col("timestamp_str"), F.lit("yyyy-MM-dd HH:mm"))
    ),
    F.when(F.col("timestamp_str").rlike("^\\d{2}/\\d{2}/\\d{4} \\d{2}:\\d{2}$"),
        F.try_to_timestamp(F.col("timestamp_str"), F.lit("dd/MM/yyyy HH:mm"))
    ),
    F.when(F.col("timestamp_str").rlike("^\\d{4}/\\d{2}/\\d{2} \\d{2}:\\d{2}$"),
        F.try_to_timestamp(F.col("timestamp_str"), F.lit("yyyy/MM/dd HH:mm"))
    ),
    F.when(F.col("timestamp_str").rlike("^\\d{2}-\\d{2}-\\d{4} \\d{2}:\\d{2}$"),
        F.try_to_timestamp(F.col("timestamp_str"), F.lit("dd-MM-yyyy HH:mm"))
    )
)

clean5 = clean4.withColumn("delivery_ts", parsed_ts).drop("timestamp_str")

ts_null_df = clean5.filter(F.col("delivery_ts").isNull())
clean6 = clean5.filter(F.col("delivery_ts").isNotNull())


In [15]:

# 10. Remove duplicate delivery IDs
clean7 = clean6.dropDuplicates(["delivery_id"])


Phase 3

In [42]:

delivered_df = clean7.filter(F.col("status") == "DELIVERED")

after_filter_count = delivered_df.count()


DateTimeException: [CANNOT_PARSE_TIMESTAMP] Text '05/01/2024 11:00' could not be parsed at index 0. Use `try_to_timestamp` to tolerate invalid input string and return NULL instead. SQLSTATE: 22007

In [41]:

print("Records before business filtering:", before_filter_count)
print("Records after keeping DELIVERED:", after_filter_count)
print("Removed due to business filter:", before_filter_count - after_filter_count)


NameError: name 'before_filter_count' is not defined

Phase 4

In [21]:

driver_df = driver_raw.select(
    F.trim(F.col("driver_id")).alias("driver_id"),
    F.trim(F.col("driver_name")).alias("driver_name"),
    F.trim(F.col("driver_level")).alias("driver_level")
)

city_zone_df = city_zone_raw.select(
    F.upper(F.trim(F.col("city"))).alias("city"),
    F.trim(F.col("zone")).alias("zone")
)


In [27]:


enriched1 = delivered_df.join(F.broadcast(driver_df), on="driver_id", how="left")
enriched = enriched1.join(F.broadcast(city_zone_df), on="city", how="left")


enriched.explain(True)


== Parsed Logical Plan ==
'Join UsingJoin(LeftOuter, [city])
:- Project [driver_id#15, delivery_id#13, city#14, status#19, delivery_time_minutes#20, delivery_ts#21, driver_name#71, driver_level#72]
:  +- Join LeftOuter, (driver_id#15 = driver_id#70)
:     :- Filter (status#19 = DELIVERED)
:     :  +- Deduplicate [delivery_id#13]
:     :     +- Filter isnotnull(delivery_ts#21)
:     :        +- Project [delivery_id#13, city#14, driver_id#15, status#19, delivery_time_minutes#20, delivery_ts#21]
:     :           +- Project [delivery_id#13, city#14, driver_id#15, timestamp_str#18, status#19, delivery_time_minutes#20, coalesce(to_timestamp(timestamp_str#18, Some(yyyy-MM-dd HH:mm), TimestampType, Some(Etc/UTC), true), to_timestamp(timestamp_str#18, Some(dd/MM/yyyy HH:mm), TimestampType, Some(Etc/UTC), true), to_timestamp(timestamp_str#18, Some(yyyy/MM/dd HH:mm), TimestampType, Some(Etc/UTC), true), to_timestamp(timestamp_str#18, Some(dd-MM-yyyy HH:mm), TimestampType, Some(Etc/UTC), true)) A

Phase 5

In [28]:

# 18. Average delivery time per city
avg_time_city = enriched.groupBy("city").agg(
    F.avg("delivery_time_minutes").alias("avg_delivery_time_minutes")
)


In [29]:

# 19. Average delivery time per driver
avg_time_driver = enriched.groupBy("driver_id", "driver_name").agg(
    F.avg("delivery_time_minutes").alias("avg_delivery_time_minutes")
)


In [31]:

 #20. Rank drivers by performance within each city (lower avg time = better rank)
driver_perf_city = enriched.groupBy("city", "driver_id", "driver_name").agg(
    F.avg("delivery_time_minutes").alias("avg_delivery_time_minutes")
)
rank_win = Window.partitionBy("city").orderBy(F.asc("avg_delivery_time_minutes"))
driver_rank_in_city = driver_perf_city.select(
    "city", "driver_id", "driver_name", "avg_delivery_time_minutes",
    F.dense_rank().over(rank_win).alias("driver_rank_in_city")
)


In [32]:

# 21. Identify fastest driver per zone (ties allowed)
driver_perf_zone = enriched.groupBy("zone", "driver_id", "driver_name").agg(
    F.avg("delivery_time_minutes").alias("avg_delivery_time_minutes")
)
zone_win = Window.partitionBy("zone").orderBy(F.asc("avg_delivery_time_minutes"))
fastest_driver_per_zone = driver_perf_zone.select(
    "zone", "driver_id", "driver_name", "avg_delivery_time_minutes",
    F.dense_rank().over(zone_win).alias("rank_in_zone")
).filter(F.col("rank_in_zone") == 1)


In [33]:

# 22. Top 2 drivers per city
top2_win = Window.partitionBy("city").orderBy(F.asc("avg_delivery_time_minutes"))
top2_drivers_per_city = driver_perf_city.select(
    "city", "driver_id", "driver_name", "avg_delivery_time_minutes",
    F.dense_rank().over(top2_win).alias("rank_in_city")
).filter(F.col("rank_in_city") <= 2)


Phase 6

In [34]:

# 23. Identify reused DataFrames: enriched is used in multiple analytics
spark.catalog.clearCache()
enriched.cache()
enriched.count()


DateTimeException: [CANNOT_PARSE_TIMESTAMP] Text '09-01-2024 12:30' could not be parsed at index 0. Use `try_to_timestamp` to tolerate invalid input string and return NULL instead. SQLSTATE: 22007

In [35]:

enriched.explain()
avg_time_city.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [city#79, driver_id#81, delivery_id#13, status#83, delivery_time_minutes#85, delivery_ts#87, driver_name#71, driver_level#72, zone#74]
   +- BroadcastHashJoin [city#79], [city#73], LeftOuter, BuildRight, false
      :- Project [driver_id#81, delivery_id#13, city#79, status#83, delivery_time_minutes#85, delivery_ts#87, driver_name#71, driver_level#72]
      :  +- BroadcastHashJoin [driver_id#81], [driver_id#70], LeftOuter, BuildRight, false
      :     :- Filter (status#83 = DELIVERED)
      :     :  +- SortAggregate(key=[delivery_id#13], functions=[first(city#14, false), first(driver_id#15, false), first(status#19, false), first(delivery_time_minutes#20, false), first(delivery_ts#21, false)])
      :     :     +- Sort [delivery_id#13 ASC NULLS FIRST], false, 0
      :     :        +- Exchange hashpartitioning(delivery_id#13, 8), ENSURE_REQUIREMENTS, [plan_id=160]
      :     :           +- SortAggregate(key=[delivery_id

In [36]:

# 26. Repartition data by city
enriched_by_city = enriched.repartition("city")
enriched_by_city.explain(True)


== Parsed Logical Plan ==
'RepartitionByExpression ['city]
+- Project [city#14, driver_id#15, delivery_id#13, status#19, delivery_time_minutes#20, delivery_ts#21, driver_name#71, driver_level#72, zone#74]
   +- Join LeftOuter, (city#14 = city#73)
      :- Project [driver_id#15, delivery_id#13, city#14, status#19, delivery_time_minutes#20, delivery_ts#21, driver_name#71, driver_level#72]
      :  +- Join LeftOuter, (driver_id#15 = driver_id#70)
      :     :- Filter (status#19 = DELIVERED)
      :     :  +- Deduplicate [delivery_id#13]
      :     :     +- Filter isnotnull(delivery_ts#21)
      :     :        +- Project [delivery_id#13, city#14, driver_id#15, status#19, delivery_time_minutes#20, delivery_ts#21]
      :     :           +- Project [delivery_id#13, city#14, driver_id#15, timestamp_str#18, status#19, delivery_time_minutes#20, coalesce(to_timestamp(timestamp_str#18, Some(yyyy-MM-dd HH:mm), TimestampType, Some(Etc/UTC), true), to_timestamp(timestamp_str#18, Some(dd/MM/yyyy HH

Phase 7

In [ ]:

# 29. Write aggregated analytics to ORC
avg_time_city.write.mode("overwrite").orc("/tmp/analytics/avg_time_city.orc")
avg_time_driver.write.mode("overwrite").orc("/tmp/analytics/avg_time_driver.orc")
driver_rank_in_city.write.mode("overwrite").orc("/tmp/analytics/driver_rank_in_city.orc")
fastest_driver_per_zone.write.mode("overwrite").orc("/tmp/analytics/fastest_driver_per_zone.orc")
top2_drivers_per_city.write.mode("overwrite").orc("/tmp/analytics/top2_drivers_per_city.orc")


In [ ]:
# 28. Write cleaned delivery data to Parquet
enriched.write.mode("overwrite").parquet("/tmp/curated/delivery_clean.parquet")

Phase 8

In [ ]:
df_ok = enriched
_ = df_ok.show(5)
